# CodeT5+ 220M — system prompt + compile-validated generation (Kaggle, 2-GPU DDP)

Adds two things to the proven 220M dual-GPU recipe: a comprehensive system prompt (covering API design,
SOLID, and the other domains actually in the dataset), and a separate compile-validated generation
mechanism as an honest substitute for true token-level grammar-constrained decoding.

## Why not true grammar-constrained decoding (researched before building this)
Checked both real options and both have a hard blocker:
- **SynCode** has a genuine built-in Python grammar, but requires `transformers>=4.53.2`, which
  directly conflicts with `transformers==4.46.3` — the pin needed just to load CodeT5's tokenizer
  without crashing (the exact bug hit and fixed twice earlier in this project).
- **transformers-CFG** is compatible with our pinned transformers, but ships example grammars for
  JSON, C, chess, and arithmetic — not Python. Writing a full Python grammar by hand is a real trap:
  Python isn't a pure context-free language because of indentation-based blocks, which is exactly the
  hard problem SynCode's own tokenizer integration exists to solve. A hand-rolled grammar, untested on
  real generation, would very likely be subtly wrong.

**What's used instead:** the project's own already-proven mechanism — beam search, then increasing-
temperature sampling retries, keeping the first candidate that compiles. This was already validated in
Phase 1 ("beam retry achieves 100% compile"). It is reported as a **separate, clearly labeled metric**
(`generate_compile_validated`, NOT pass@1) — never conflated with the honest single-pass correctness
numbers (`generate_texts_greedy`), to avoid repeating the PyCodeGPT multi-sample-vs-pass@1 mistake from
earlier in this project.

## The system prompt
Applied identically to every example, every task, training AND eval (HumanEval/ClassEval included) —
a system block that differs between training and eval would teach the model to key off a signal that
then changes at test time, which is worse than having none. Explicitly mentions API design, SOLID
principles, and the other domains present in the training data, per your request. Kept to ~45 words
since it's prepended to every single row and eats into the 512-token input budget; **Cell 3 measures
the real token-fit cost on your actual data before training starts**, rather than assuming it's fine.

## Before running (on kaggle.com)
1. `dataset_formats_v2.parquet` attached via **"Add Input"**.
2. Settings -> Accelerator -> **GPU T4 x2**.
3. Settings -> Internet -> **On**.

## Everything else, unchanged from the proven recipe
Model `Salesforce/codet5p-220m`, format `tgt_b_canonical`, the 7 task-marker special tokens with
mean-initialization (not random), true DDP via `notebook_launcher` (not `DataParallel`), 7 max epochs
with early stopping, lr 2e-5, the same subprocess-isolated HumanEval(64)/ClassEval(10) harness proven
against reference solutions.

## Important
**Not executed end to end** here (real 2-GPU training with live model downloads isn't something this
delivery environment can run). Every cell is syntax-validated. Run it and read Cell 10's comparison
against your two prior runs (CodeT5-small baseline, plain CodeT5+ 220M).

In [ ]:
# CELL 0 — INSTALL (pinned -- see the markdown above for exactly why)
!pip install -q "transformers==4.46.3" "accelerate>=0.26.0" datasets pandas pyarrow
print('Installed. RESTART THE SESSION NOW (Run -> Restart Session), then run Cell 1 onward.')

In [ ]:
# CELL 1 — IMPORTS + CONFIG
# CUDA-safety note (same as the earlier dual-GPU notebook): nothing here may touch CUDA before
# Cell 5's notebook_launcher runs -- no .cuda(), no .to("cuda"), no model loading at top level.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # must be set BEFORE torch import

import ast, copy, gc, hashlib, json, math, random, re, signal, subprocess, sys, tempfile, time, warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore", category=SyntaxWarning)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

PARQUET_FILENAME = "dataset_formats_v2.parquet"
KAGGLE_INPUT_DIR = Path("/kaggle/input")
WORK = Path("/kaggle/working/codet5p_220m_v2")
WORK.mkdir(parents=True, exist_ok=True)
BEST_MODEL_DIR = WORK / "best_model"
RESULTS_FILE = WORK / "results.json"

BASE_MODEL = "Salesforce/codet5p-220m"
TARGET_FORMAT_COL = "tgt_b_canonical"
FITS_COL = "fits512_b"
NEW_SPECIAL_TOKENS = ["<GENERATE>", "<REPAIR>", "<REFACTOR>", "<CODE>", "</CODE>", "<ERROR>", "</ERROR>"]

# The system block. Applied identically to every example, every task, training AND eval (HumanEval/
# ClassEval included) -- a system block that isn't byte-identical between training and eval would
# teach the model to key off a signal that then changes at test time, which is worse than having none.
# Kept to ~45 words (roughly 60-80 tokens with this tokenizer) since it is prepended to EVERY example
# and eats into the 512-token input budget on every single row -- see Cell 3 for the real token-fit
# numbers after adding this, measured rather than assumed.
SYSTEM_PROMPT = (
    "You are a Python code generation assistant for software engineering tasks spanning API design, "
    "database access, data structures and algorithms, and data science code. Rules: output only "
    "complete, syntactically valid Python with every import included. Never include explanations or "
    "text outside the code. Fully implement every function, method, and class -- never stop early or "
    "leave a placeholder. Where relevant, follow SOLID principles, RESTful API conventions, and "
    "proper error handling."
)

EPOCHS = 7
LR = 2e-5
BATCH_SIZE = 8            # PER GPU
GRAD_ACCUM = 2
MAX_SRC_LEN = 512
MAX_TGT_LEN = 512

HE_N = 64
CLASSEVAL_N = 10
CLASSEVAL_MAX_PROMPT_TOKENS = 450
EVAL_OWN_N = 300
GEN_BATCH = 16
RETRY_TEMPERATURES = [0.2, 0.5, 0.8]   # used ONLY by the separate "compile-validated" pass, see Cell 9

N_GPUS = torch.cuda.device_count()
print(f"torch {torch.__version__} | GPUs visible: {N_GPUS}")
if N_GPUS == 0:
    raise RuntimeError(
        "No GPU found. In the right sidebar: Settings -> Accelerator -> GPU T4 x2, and Internet -> On. "
        "The session restarts; then run this cell again."
    )
print(f"Base model: {BASE_MODEL} | format: {TARGET_FORMAT_COL} | epochs: {EPOCHS}")
print(f"System prompt ({len(SYSTEM_PROMPT.split())} words): {SYSTEM_PROMPT}")

## Load the dataset

In [ ]:
# CELL 2 — FIND + LOAD dataset_formats_v2.parquet
candidates = list(KAGGLE_INPUT_DIR.rglob(PARQUET_FILENAME))
if not candidates:
    print(f"Could not find {PARQUET_FILENAME} under {KAGGLE_INPUT_DIR}. What's attached:")
    if KAGGLE_INPUT_DIR.exists():
        for p in KAGGLE_INPUT_DIR.rglob("*"):
            if p.is_file():
                print(f"  {p}")
    raise FileNotFoundError("Attach the dataset with this file via 'Add Input', then re-run this cell.")
src = candidates[0]
print(f"Using: {src} ({src.stat().st_size/1e6:.1f} MB)")

df = pd.read_parquet(src)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")

need = ["id", "unit_id", "split", "task", "instruction", "context_code", "error",
        TARGET_FORMAT_COL, FITS_COL, "ok_b"]
missing = [c for c in need if c not in df.columns]
assert not missing, f"expected columns missing: {missing}"

mask = df[FITS_COL].astype(bool) & df["ok_b"].astype(bool)
before = len(df)
df = df[mask].reset_index(drop=True)
print(f"Kept rows with a valid, within-budget {TARGET_FORMAT_COL}: {before} -> {len(df)}")
print("tasks:", dict(df.task.value_counts()))

## Rebuild input_text with the system prompt, measure the real token-fit cost

In [ ]:
# CELL 3 — REBUILD input_text WITH THE SYSTEM PROMPT, MEASURE THE REAL TOKEN-FIT COST
# The Parquet's precomputed "input_text" column does NOT include the system prompt -- it's rebuilt
# here from the raw instruction/context_code/error fields so the system block is applied identically
# to every row, training and eval alike.
def build_input_text(row):
    task = row["task"]
    if task == "generate":
        body = f"<GENERATE> {row['instruction']}"
    elif task == "repair":
        body = (f"<REPAIR> {row['instruction']}\n<CODE>\n{row.get('context_code','')}\n</CODE>"
                f"\n<ERROR>\n{row.get('error','')}\n</ERROR>")
    elif task == "refactor":
        body = f"<REFACTOR> {row['instruction']}\n<CODE>\n{row.get('context_code','')}\n</CODE>"
    else:
        raise ValueError(f"unknown task: {task}")
    return f"{SYSTEM_PROMPT}\n\n{body}"

df["input_text"] = df.apply(build_input_text, axis=1)

# Measure the real cost -- do this BEFORE training, not after, so a budget problem is caught early.
from transformers import AutoTokenizer
_tok_probe = AutoTokenizer.from_pretrained(BASE_MODEL)
sys_len = len(_tok_probe(SYSTEM_PROMPT)["input_ids"])
print(f"System prompt alone: {sys_len} tokens")

sample_lens = [len(_tok_probe(t, truncation=False)["input_ids"]) for t in df["input_text"].sample(min(1000, len(df)), random_state=SEED)]
fits_512 = sum(1 for l in sample_lens if l <= MAX_SRC_LEN) / len(sample_lens) * 100
print(f"Rows whose input (system prompt + task text) fits {MAX_SRC_LEN} tokens: {fits_512:.1f}% "
      f"(measured on a {len(sample_lens)}-row sample)")
print(f"Mean input tokens with system prompt: {np.mean(sample_lens):.0f} | median: {np.median(sample_lens):.0f}")

by_task_fit = {}
for task in ("generate", "repair", "refactor"):
    sub = df[df.task == task]["input_text"].sample(min(300, (df.task == task).sum()), random_state=SEED)
    lens = [len(_tok_probe(t, truncation=False)["input_ids"]) for t in sub]
    by_task_fit[task] = sum(1 for l in lens if l <= MAX_SRC_LEN) / len(lens) * 100
    print(f"  {task:9s} fits {MAX_SRC_LEN}: {by_task_fit[task]:.1f}%  (mean {np.mean(lens):.0f} tokens)")
if min(by_task_fit.values()) < 85:
    print("\nWARNING: the system prompt is pushing some task's inputs noticeably over budget "
          "(truncation=True below will silently cut the END of the input, which for repair/refactor "
          "could cut off the actual code or error). Consider shortening SYSTEM_PROMPT in Cell 1 if this")
    print("number looks concerning for your use case.")
del _tok_probe

## Define the training function (runs once per GPU -- not called yet)

In [ ]:
# CELL 4 — DEFINE THE TRAINING FUNCTION (runs once per GPU, launched by Cell 5's notebook_launcher)
def train_codet5p_220m():
    import torch
    from transformers import (
        AutoTokenizer, T5ForConditionalGeneration,
        Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, EarlyStoppingCallback,
    )
    from datasets import Dataset
    from accelerate import PartialState

    state = PartialState()
    is_main = state.is_main_process
    if is_main:
        print(f"[rank {state.process_index}] starting on {state.num_processes} process(es)")

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    vocab_before = len(tokenizer)
    tokenizer.add_special_tokens({"additional_special_tokens": NEW_SPECIAL_TOKENS})
    if is_main:
        print(f"Tokenizer vocab: {vocab_before} -> {len(tokenizer)}")

    model = T5ForConditionalGeneration.from_pretrained(BASE_MODEL)
    old_embeddings = model.get_input_embeddings().weight.data.clone()
    mean_embedding = old_embeddings.mean(dim=0)
    model.resize_token_embeddings(len(tokenizer))
    with torch.no_grad():
        input_emb = model.get_input_embeddings()
        for t in NEW_SPECIAL_TOKENS:
            tid = tokenizer.convert_tokens_to_ids(t)
            input_emb.weight.data[tid] = mean_embedding.clone()
    if is_main:
        print(f"New special tokens mean-initialized. tie_word_embeddings={model.config.tie_word_embeddings}")
        total_p = sum(p.numel() for p in model.parameters())
        print(f"Total params: {total_p/1e6:.1f}M")

    def tokenize_split(frame, desc):
        inputs = tokenizer(frame["input_text"].tolist(), truncation=True, max_length=MAX_SRC_LEN)
        targets = tokenizer(frame[TARGET_FORMAT_COL].tolist(), truncation=True, max_length=MAX_TGT_LEN)
        ds = Dataset.from_dict({"input_ids": inputs["input_ids"], "attention_mask": inputs["attention_mask"],
                                "labels": targets["input_ids"]})
        if is_main:
            print(f"{desc}: {len(ds)} examples tokenized")
        return ds

    train_df_local = df[df.split == "train"].reset_index(drop=True)
    val_df_local = df[df.split == "val"].reset_index(drop=True)
    train_ds = tokenize_split(train_df_local, "train")
    val_ds = tokenize_split(val_df_local, "val")
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

    args = Seq2SeqTrainingArguments(
        output_dir=str(WORK / "checkpoints"),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        warmup_ratio=0.1,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        fp16=True,
        gradient_checkpointing=True,
        logging_steps=20,
        report_to="none",
        seed=SEED,
        ddp_find_unused_parameters=False,
    )

    trainer = Seq2SeqTrainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=data_collator, processing_class=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    if is_main:
        print(f"Training: {EPOCHS} max epochs, lr={LR}, per-GPU batch={BATCH_SIZE}x{GRAD_ACCUM}, "
              f"{state.num_processes} GPU(s) via real DDP")
    train_result = trainer.train()

    if is_main:
        print(f"\nTraining complete. {train_result}")
        trainer.save_model(str(BEST_MODEL_DIR))
        tokenizer.save_pretrained(str(BEST_MODEL_DIR))
        print(f"Saved best checkpoint to {BEST_MODEL_DIR}")
        history = [{k: v for k, v in log.items() if k in ("epoch", "loss", "eval_loss")}
                   for log in trainer.state.log_history if "eval_loss" in log or "loss" in log]
        (WORK / "train_history.json").write_text(json.dumps(history, indent=1))
        print(f"Saved training history ({len(history)} log entries)")

print("Training function defined (not yet run -- Cell 5 launches it on all visible GPUs).")

## Launch true 2-GPU DDP training

In [ ]:
# CELL 5 — LAUNCH TRUE DDP TRAINING ACROSS ALL VISIBLE GPUs
from accelerate import notebook_launcher

t0 = time.time()
notebook_launcher(train_codet5p_220m, num_processes=N_GPUS)
print(f"\nTotal training wall-clock time: {(time.time()-t0)/60:.1f} minutes")

assert BEST_MODEL_DIR.exists(), "training did not produce a saved model -- check the output above for errors"
print(f"Best model confirmed on disk at {BEST_MODEL_DIR}")

## Reload the trained model for evaluation (single GPU, main process)

In [ ]:
# CELL 6 — RELOAD THE TRAINED MODEL IN THIS (MAIN) PROCESS FOR EVALUATION
from transformers import AutoTokenizer, T5ForConditionalGeneration

DEVICE = "cuda:0"
tokenizer = AutoTokenizer.from_pretrained(str(BEST_MODEL_DIR))
model = T5ForConditionalGeneration.from_pretrained(str(BEST_MODEL_DIR)).to(DEVICE)
model.eval()
print(f"Reloaded trained model from {BEST_MODEL_DIR} onto {DEVICE}")
for t in NEW_SPECIAL_TOKENS:
    tid = tokenizer.convert_tokens_to_ids(t)
    assert tid != tokenizer.unk_token_id, f"{t} did not survive the save/reload as a real token"
print("Special tokens confirmed present after reload.")

## Evaluation kit (subprocess-isolated, hard timeouts, proven against reference solutions)

In [ ]:
# CELL 7 — EVALUATION KIT (subprocess-isolated, hard timeouts, proven against reference solutions)
PRELUDE = ("import resource\n"
           "try:\n    resource.setrlimit(resource.RLIMIT_AS, (3 << 30, 3 << 30))\nexcept Exception:\n    pass\n")

def run_python(files, entry, args=(), timeout=10):
    with tempfile.TemporaryDirectory() as d:
        for name, text in files.items():
            Path(d, name).write_text((PRELUDE + text) if name == entry else text, encoding="utf-8")
        proc = subprocess.Popen([sys.executable, entry, *map(str, args)], cwd=d, stdin=subprocess.DEVNULL,
                                stdout=subprocess.PIPE, stderr=subprocess.PIPE, encoding="utf-8",
                                errors="replace", start_new_session=True)
        try:
            out, err = proc.communicate(timeout=timeout)
            return proc.returncode, out, err
        except subprocess.TimeoutExpired:
            try:
                os.killpg(proc.pid, signal.SIGKILL)
            except Exception:
                pass
            try:
                proc.communicate(timeout=5)
            except Exception:
                pass
            return -9, "", "timeout"

HE_RUNNER = '''
import json, sys
spec = json.load(open("spec.json"))
src = open("prog.py", encoding="utf-8").read()
ns = {"__name__": "heval_module"}
try:
    exec(compile(src, "prog.py", "exec"), ns)
    ns["check"](ns[spec["entry"]])
except BaseException as e:
    sys.stderr.write(type(e).__name__ + ": " + str(e)[:300] + "\\n")
    sys.exit(1)
print("__PASS__")
'''

CE_RUNNER = '''
import io, json, sys, unittest
src = open("gen.py", encoding="utf-8").read() + "\\n\\n" + open("test.py", encoding="utf-8").read()
ns = {"__name__": "classeval_module"}
try:
    exec(compile(src, "prog.py", "exec"), ns)
except BaseException as e:
    print(json.dumps({"status": "exec_fail", "error": (type(e).__name__ + ": " + str(e))[:200]})); sys.exit(0)
classes = [v for v in ns.values() if isinstance(v, type) and issubclass(v, unittest.TestCase) and v is not unittest.TestCase]
total = failed = 0
for cls in classes:
    res = unittest.TextTestRunner(stream=io.StringIO(), verbosity=0).run(unittest.defaultTestLoader.loadTestsFromTestCase(cls))
    total += res.testsRun
    failed += len(res.failures) + len(res.errors)
print(json.dumps({"status": "ok", "tests": total, "failed": failed}))
'''

def top_level_imports(src):
    try:
        tree = ast.parse(src)
    except SyntaxError:
        return "\n".join(l for l in src.splitlines() if l.startswith(("import ", "from ")))
    keep = [n for n in tree.body if isinstance(n, (ast.Import, ast.ImportFrom))]
    return ast.unparse(ast.Module(body=keep, type_ignores=[])) if keep else ""

def humaneval_header(prompt, entry):
    try:
        tree = ast.parse(prompt)
        keep = [n for n in tree.body if isinstance(n, (ast.Import, ast.ImportFrom))
                or (isinstance(n, ast.FunctionDef) and n.name != entry)]
        return ast.unparse(ast.Module(body=keep, type_ignores=[])) if keep else ""
    except SyntaxError:
        return top_level_imports(prompt)

def run_humaneval(prompt, entry, test, gen_src, timeout=20):
    prog = humaneval_header(prompt, entry) + "\n\n" + gen_src + "\n\n" + test + "\n"
    rc, out, err = run_python({"runner.py": HE_RUNNER, "prog.py": prog, "spec.json": json.dumps({"entry": entry})},
                              "runner.py", timeout=timeout)
    return rc == 0 and "__PASS__" in out

def run_classeval(skeleton, test, gen_src, timeout=30):
    gen = top_level_imports(skeleton) + "\n\n" + gen_src
    rc, out, err = run_python({"runner.py": CE_RUNNER, "gen.py": gen, "test.py": test}, "runner.py", timeout=timeout)
    if rc == -9:
        return {"status": "timeout", "tests": 0, "failed": 0}
    try:
        return json.loads(out.strip().splitlines()[-1])
    except Exception:
        return {"status": "runner_error", "tests": 0, "failed": 0}

def is_valid_python(code_str):
    try:
        tree = ast.parse(code_str)
    except Exception:
        return False
    try:
        compile(tree, "<e>", "exec")
        return True
    except Exception:
        return False

def defines(src, name, kind):
    try:
        return any(isinstance(n, kind) and n.name == name for n in ast.walk(ast.parse(src)))
    except Exception:
        return False

def canonical_humaneval_src(row):
    tree = ast.parse(row["prompt"] + row["canonical_solution"])
    node = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == row["entry_point"])
    return ast.unparse(node)

def load_humaneval_rows(source):
    if source == "hf":
        from datasets import load_dataset
        return [dict(r) for r in load_dataset("evalplus/humanevalplus", split="test")]
    import gzip, urllib.request
    raw = urllib.request.urlopen("https://raw.githubusercontent.com/openai/human-eval/master/data/HumanEval.jsonl.gz", timeout=60).read()
    return [json.loads(l) for l in gzip.decompress(raw).decode("utf-8").splitlines() if l.strip()]

def load_classeval_rows(source):
    if source == "hf":
        from datasets import load_dataset
        return [dict(r) for r in load_dataset("FudanSELab/ClassEval", split="test")]
    import urllib.request
    return json.loads(urllib.request.urlopen("https://raw.githubusercontent.com/FudanSELab/ClassEval/master/data/ClassEval_data.json", timeout=60).read().decode("utf-8"))

def select_humaneval(n, seed):
    best = None
    for source in ("hf", "github"):
        try:
            rows = load_humaneval_rows(source)
        except Exception as e:
            print(f"  HumanEval from {source}: unavailable ({type(e).__name__}: {str(e)[:80]})")
            continue
        idx = list(range(len(rows)))
        random.Random(seed).shuffle(idx)
        step = len(idx) / n
        chosen = [rows[i] for i in sorted(idx[int(k * step)] for k in range(n))]
        good = []
        for r in chosen:
            try:
                ok = run_humaneval(r["prompt"], r["entry_point"], r["test"], canonical_humaneval_src(r), timeout=20)
            except Exception as e:
                print(f"    reference-check error on {r.get('task_id')}: {type(e).__name__}: {str(e)[:100]}")
                ok = False
            if ok:
                good.append(r)
        print(f"  HumanEval from {source}: {len(good)}/{len(chosen)} reference solutions pass the harness")
        if best is None or len(good) > len(best[1]):
            best = (source, good, len(chosen))
        if len(good) >= 0.9 * len(chosen):
            break
    return best

def select_classeval(n, seed, tok):
    for source in ("hf", "github"):
        try:
            rows = load_classeval_rows(source)
            break
        except Exception as e:
            print(f"  ClassEval from {source}: unavailable ({type(e).__name__}: {str(e)[:80]})")
    else:
        return None
    fits = [r for r in rows if len(tok(f"<GENERATE> Complete this Python class:\n{r['skeleton']}")["input_ids"]) <= CLASSEVAL_MAX_PROMPT_TOKENS]
    random.Random(seed).shuffle(fits)
    chosen, tried = [], 0
    for r in fits:
        if len(chosen) >= n or tried >= 60:
            break
        tried += 1
        try:
            res = run_classeval(r["skeleton"], r["test"], r["solution_code"], timeout=30)
        except Exception as e:
            print(f"    reference-check error on {r.get('task_id')}: {type(e).__name__}: {str(e)[:100]}")
            res = {"status": "error"}
        if res.get("status") == "ok" and res.get("tests", 0) > 0 and res.get("failed", 1) == 0:
            chosen.append(r)
    print(f"  ClassEval from {source}: {len(rows)} problems, {len(fits)} fit the encoder window, "
          f"tried {tried}, kept {len(chosen)} whose reference solution passes the harness")
    return source, chosen

print("Evaluation kit ready (isolated processes, hard time limits, tests must actually run).")

## Generation: greedy pass@1 (primary) + compile-validated (separate, not pass@1)

In [ ]:
# CELL 8 — GENERATION (two SEPARATE, clearly labeled mechanisms -- never conflate the two)
#
#  generate_texts_greedy()     -> TRUE pass@1: single pass, no beams, no sampling. This is the
#                                 primary, honest correctness number, same standard used throughout
#                                 this project (PyCodeGPT's inflated 100-sample pass@k was the
#                                 cautionary example for why this distinction matters).
#
#  generate_compile_validated() -> NOT pass@1. Tries beam search, then increasing-temperature
#                                 sampling, keeping the first candidate that actually compiles.
#                                 This is the project's already-proven mechanism for "make the
#                                 OUTPUT syntactically valid" (confirmed in Phase 1: "beam retry
#                                 achieves 100% compile"). It answers "can SOME attempt produce
#                                 valid Python", not "is the content correct" -- report it
#                                 separately, never as a correctness metric.

@torch.no_grad()
def generate_texts_greedy(mdl, texts, max_new_tokens=MAX_TGT_LEN, batch_size=GEN_BATCH):
    mdl.eval()
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    outs = [None] * len(texts)
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        enc = tokenizer([texts[i] for i in idx], return_tensors="pt", padding=True,
                        truncation=True, max_length=MAX_SRC_LEN).to(DEVICE)
        with torch.amp.autocast("cuda", enabled=True):
            seq = mdl.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                               max_new_tokens=max_new_tokens, num_beams=1, do_sample=False)
        for j, i in enumerate(idx):
            outs[i] = tokenizer.decode(seq[j], skip_special_tokens=True)
    return outs

@torch.no_grad()
def generate_compile_validated(mdl, text, max_new_tokens=MAX_TGT_LEN, num_beams=4):
    """One input at a time (not batched -- this does up to 4 separate generation calls per input,
    so it is meaningfully slower; only use it on the smaller benchmark sets, not the full test split)."""
    enc = tokenizer([text], return_tensors="pt", truncation=True, max_length=MAX_SRC_LEN).to(DEVICE)
    with torch.amp.autocast("cuda", enabled=True):
        seq = mdl.generate(**enc, max_new_tokens=max_new_tokens, num_beams=num_beams, early_stopping=True)
    out = tokenizer.decode(seq[0], skip_special_tokens=True)
    if is_valid_python(out):
        return out, "beam"
    for temp in RETRY_TEMPERATURES:
        with torch.amp.autocast("cuda", enabled=True):
            seq2 = mdl.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True, temperature=temp, top_p=0.95)
        out2 = tokenizer.decode(seq2[0], skip_special_tokens=True)
        if is_valid_python(out2):
            return out2, f"sample_t{temp}"
    return out, "beam_invalid"   # nothing compiled -- return the beam result, labeled as still invalid

print("Generation functions ready: generate_texts_greedy (pass@1) and generate_compile_validated (NOT pass@1).")

## Run the full evaluation

In [ ]:
# CELL 9 — RUN THE FULL EVALUATION (greedy pass@1 as primary, compile-validated as a separate,
# clearly labeled secondary measurement)
summary = {}

print("Selecting benchmark problems (reference solutions run through the harness first)...")
he_sel = None
try:
    he_sel = select_humaneval(HE_N, SEED)
except Exception as e:
    print(f"  HumanEval selection failed ({type(e).__name__}: {str(e)[:150]})")
he_problems = he_sel[1] if he_sel else []

ce_sel = None
try:
    ce_sel = select_classeval(CLASSEVAL_N, SEED, tokenizer)
except Exception as e:
    print(f"  ClassEval selection failed ({type(e).__name__}: {str(e)[:150]})")
ce_problems = ce_sel[1] if ce_sel else []
print(f"\nUsable: HumanEval {len(he_problems)}/{HE_N}  ClassEval {len(ce_problems)}/{CLASSEVAL_N}")

# ---------- own-dataset: greedy pass@1 (primary) ----------
random.seed(SEED)
test_df = df[df.split == "test"].reset_index(drop=True)
eval_rows = test_df.sample(min(EVAL_OWN_N, len(test_df)), random_state=SEED).reset_index(drop=True)
print(f"\nGenerating {len(eval_rows)} own-dataset outputs (greedy, pass@1)...")
t0 = time.time()
gens = generate_texts_greedy(model, eval_rows["input_text"].tolist())
gen_sec = time.time() - t0
print(f"  done in {gen_sec:.0f}s")

def norm_dump(code_str):
    tree = ast.parse(code_str)
    for n in ast.walk(tree):
        if isinstance(n, ast.Constant):
            n.kind = None
        if hasattr(n, "type_comment"):
            n.type_comment = None
    return ast.dump(tree)

def exact_match(pred_src, ref_src):
    try:
        return norm_dump(pred_src) == norm_dump(ref_src)
    except Exception:
        return False

own_records = []
for (_, row), text in zip(eval_rows.iterrows(), gens):
    valid = is_valid_python(text)
    exact = exact_match(text, row[TARGET_FORMAT_COL]) if valid else False
    own_records.append({"id": row["id"], "task": row["task"], "valid": valid, "exact": exact})
own_df = pd.DataFrame(own_records)

summary["own_greedy"] = {
    "n": len(own_df), "valid": int(own_df.valid.sum()), "exact": int(own_df.exact.sum()), "gen_seconds": gen_sec,
    "by_task": {t: {"n": len(g), "valid": int(g.valid.sum()), "exact": int(g.exact.sum())}
                for t, g in own_df.groupby("task")},
}
print(f"Own-dataset greedy: valid {summary['own_greedy']['valid']}/{summary['own_greedy']['n']}  "
      f"exact {summary['own_greedy']['exact']}/{summary['own_greedy']['n']}")
for t, s in summary["own_greedy"]["by_task"].items():
    print(f"  {t:9s} valid {s['valid']}/{s['n']}  exact {s['exact']}/{s['n']}")

# ---------- HumanEval: greedy pass@1 (primary metric) ----------
he_passed = 0
if he_problems:
    gens_he = generate_texts_greedy(model, [f"<GENERATE> Complete this Python function:\n{r['prompt']}" for r in he_problems])
    for r, text in zip(he_problems, gens_he):
        has_def = is_valid_python(text) and defines(text, r["entry_point"], ast.FunctionDef)
        passed = run_humaneval(r["prompt"], r["entry_point"], r["test"], text) if has_def else False
        he_passed += bool(passed)
    print(f"\nHumanEval pass@1 (greedy): {he_passed}/{len(he_problems)} ({he_passed/len(he_problems)*100:.1f}%)")
else:
    print("\nHumanEval: skipped (no usable problems)")
summary["humaneval"] = {"n": len(he_problems), "passed": he_passed}

# ---------- ClassEval: greedy pass@1 (primary metric) ----------
ce_passed = 0
if ce_problems:
    gens_ce = generate_texts_greedy(model, [f"<GENERATE> Complete this Python class:\n{r['skeleton']}" for r in ce_problems])
    for r, text in zip(ce_problems, gens_ce):
        has_cls = is_valid_python(text) and defines(text, r["class_name"], ast.ClassDef)
        res = run_classeval(r["skeleton"], r["test"], text) if has_cls else {"status": "no_class", "tests": 0, "failed": 0}
        passed = res.get("status") == "ok" and res.get("tests", 0) > 0 and res.get("failed", 1) == 0
        ce_passed += bool(passed)
    print(f"ClassEval class-level pass@1 (greedy): {ce_passed}/{len(ce_problems)} ({ce_passed/len(ce_problems)*100:.1f}%)")
else:
    print("ClassEval: skipped (no usable problems)")
summary["classeval"] = {"n": len(ce_problems), "passed": ce_passed}

# ---------- SEPARATE SECTION: compile-validated generation (NOT pass@1) ----------
# Only run on the smaller benchmark sets (HumanEval + ClassEval), since each input costs up to
# 4 separate generation calls -- too slow to run on the full 300-row own-dataset sample.
print("\n" + "=" * 70)
print("COMPILE-VALIDATED GENERATION (beam + retry-until-valid -- NOT a pass@1/correctness metric)")
print("=" * 70)

he_cv_valid = 0
if he_problems:
    for r in he_problems:
        text, method = generate_compile_validated(model, f"<GENERATE> Complete this Python function:\n{r['prompt']}")
        he_cv_valid += is_valid_python(text)
    print(f"HumanEval prompts: valid Python under compile-validated generation: {he_cv_valid}/{len(he_problems)} "
          f"({he_cv_valid/len(he_problems)*100:.1f}%)  [compare against the greedy valid-Python rate, not pass@1]")
summary["humaneval_compile_validated_valid_pct"] = (he_cv_valid / len(he_problems) * 100) if he_problems else None

ce_cv_valid = 0
if ce_problems:
    for r in ce_problems:
        text, method = generate_compile_validated(model, f"<GENERATE> Complete this Python class:\n{r['skeleton']}")
        ce_cv_valid += is_valid_python(text)
    print(f"ClassEval prompts: valid Python under compile-validated generation: {ce_cv_valid}/{len(ce_problems)} "
          f"({ce_cv_valid/len(ce_problems)*100:.1f}%)")
summary["classeval_compile_validated_valid_pct"] = (ce_cv_valid / len(ce_problems) * 100) if ce_problems else None

RESULTS_FILE.write_text(json.dumps(summary, indent=2, default=str))
print(f"\nSaved: {RESULTS_FILE}")

## Report — compare against the two prior runs

In [ ]:
# CELL 10 — REPORT: DID THE SYSTEM PROMPT HELP? DOES COMPILE-VALIDATION CLOSE THE VALIDITY GAP?
history = json.loads((WORK / "train_history.json").read_text())
summary = json.loads(RESULTS_FILE.read_text())

print("=" * 90)
print("CodeT5+ 220M + SYSTEM PROMPT + compile-validated generation (DDP, 2 GPUs)")
print("=" * 90)

val_losses = [h["eval_loss"] for h in history if "eval_loss" in h]
print(f"\nValidation loss by epoch: {' -> '.join(f'{v:.3f}' for v in val_losses)}")

og = summary["own_greedy"]
print(f"\nOwn-dataset, GREEDY pass@1 (n={og['n']}):")
print(f"  valid Python: {og['valid']}/{og['n']} ({og['valid']/og['n']*100:.1f}%)")
print(f"  exact match:  {og['exact']}/{og['n']} ({og['exact']/og['n']*100:.1f}%)")
for t, s in og["by_task"].items():
    print(f"    {t:9s} valid {s['valid']}/{s['n']} ({s['valid']/s['n']*100:.1f}%)   "
          f"exact {s['exact']}/{s['n']} ({s['exact']/s['n']*100:.1f}%)")

he, ce = summary["humaneval"], summary["classeval"]
print(f"\nHumanEval pass@1 (greedy): {he['passed']}/{he['n']}" + (f" ({he['passed']/he['n']*100:.1f}%)" if he["n"] else " (skipped)"))
print(f"ClassEval pass@1 (greedy): {ce['passed']}/{ce['n']}" + (f" ({ce['passed']/ce['n']*100:.1f}%)" if ce["n"] else " (skipped)"))

print(f"\nCompile-validated valid-Python rate (NOT pass@1 -- compare to greedy's valid% above):")
hv = summary.get("humaneval_compile_validated_valid_pct")
cv = summary.get("classeval_compile_validated_valid_pct")
print(f"  HumanEval prompts: {hv:.1f}%" if hv is not None else "  HumanEval: skipped")
print(f"  ClassEval prompts: {cv:.1f}%" if cv is not None else "  ClassEval: skipped")

print("\n" + "=" * 90)
print("COMPARISON AGAINST THE TWO EARLIER RUNS (same dataset, same format, no system prompt)")
print("=" * 90)
print(f"{'metric':<28} {'CodeT5-small':>14} {'CodeT5+220M':>14} {'CodeT5+220M+sys':>16}")
print(f"{'generate valid %':<28} {'68.3':>14} {'90.8':>14} {og['by_task'].get('generate',{}).get('valid',0)/max(1,og['by_task'].get('generate',{}).get('n',1))*100:>16.1f}")
print(f"{'generate exact %':<28} {'0.0':>14} {'0.0':>14} {og['by_task'].get('generate',{}).get('exact',0)/max(1,og['by_task'].get('generate',{}).get('n',1))*100:>16.1f}")
print(f"{'repair exact %':<28} {'16.8':>14} {'76.8':>14} {og['by_task'].get('repair',{}).get('exact',0)/max(1,og['by_task'].get('repair',{}).get('n',1))*100:>16.1f}")
print(f"{'HumanEval %':<28} {'0.0':>14} {'0.0':>14} {(he['passed']/he['n']*100 if he['n'] else 0):>16.1f}")
print(f"{'ClassEval %':<28} {'0.0':>14} {'0.0':>14} {(ce['passed']/ce['n']*100 if ce['n'] else 0):>16.1f}")

print("\n" + "=" * 90)
print("VERDICT")
print("=" * 90)
gen_stats = og["by_task"].get("generate")
he_pct = (he["passed"] / he["n"] * 100) if he["n"] else 0
if gen_stats and (gen_stats["exact"] / gen_stats["n"] * 100 > 1 or he_pct > 1):
    print("Generate moved off zero with the system prompt + domain hints. This is evidence the task")
    print("framing (not just raw model size) was part of the bottleneck. Worth scaling this recipe up.")
elif hv is not None and hv > og["valid"] / og["n"] * 100 + 5:
    print(f"Content correctness (exact match, HumanEval) did NOT improve, but compile-validated generation")
    print(f"raised the valid-Python rate meaningfully ({hv:.1f}% vs {og['valid']/og['n']*100:.1f}% greedy).")
    print("This confirms compile-validation closes the SYNTAX gap as expected -- it does not and was never")
    print("expected to fix the underlying LOGIC/correctness gap. That gap is still unresolved.")
else:
    print("Neither the system prompt nor compile-validated generation moved generate's correctness numbers")
    print("(exact match, HumanEval) off zero. Combined with the earlier CodeT5-small vs CodeT5+ 220M result")
    print("(3.7x size gave zero improvement on these same numbers), this is now three different levers")
    print("(size, framing, output-validity) that did not touch the core problem. Strongest remaining")
    print("hypothesis: try a genuinely different, instruction-tuned architecture (e.g.")
    print("Qwen2.5-Coder-0.5B-Instruct) before concluding anything further about the T5 family specifically.")

## Finish (download from Kaggle's Output tab)

In [ ]:
# CELL 11 — FINISH (Kaggle: download from the Output tab)
for f in (RESULTS_FILE, WORK / "train_history.json"):
    print(f"{f}  ({f.stat().st_size/1e3:.0f} KB)")
print(f"Model checkpoint: {BEST_MODEL_DIR}  (saved, large -- see note below)")
print("\nTo get these files onto your computer:")
print("  1. Click 'Save Version' (top right) -> 'Save & Run All' -> Save.")
print("  2. Once it finishes, open this notebook's 'Output' tab.")
print("  3. results.json and train_history.json will be listed there with a Download button.")
print("  Note: the saved model checkpoint directory is large; download only results.json and")
print("  train_history.json unless you specifically need the model weights.")